# 第52章 热力图（heatmap）

用颜色矩阵展示相关系数、透视表或任意二维数值。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。


## 适用场景

比较行×列组合或压缩读取数值矩阵。

## 数据结构

二维矩阵或可透视成长×宽矩阵的长表。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 cmap="vlag" 改为 cmap="coolwarm" 或 "RdBu_r"，对比不同发散色盘的视觉效果
2. 修改 center=0 为不设置 center，观察色阶中心对相关矩阵显示的影响
3. 调整 fmt=".2f" 为 fmt=".0f"，说明标注精度对数值可读性的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(36)
n = 240
orders = pd.DataFrame({
    "category": rng.choice(["办公", "数码", "家居"], n, p=[0.34, 0.38, 0.28]),
    "channel": rng.choice(["自然流量", "广告", "会员"], n, p=[0.42, 0.36, 0.22]),
    "region": rng.choice(["华东", "华南", "华北"], n),
    "order_value": np.clip(rng.normal(260, 72, n), 45, None),
    "items": rng.integers(1, 7, n),
})
orders.loc[orders["category"] == "数码", "order_value"] *= 1.35
orders["satisfied"] = rng.choice(["满意", "一般"], n, p=[0.78, 0.22])

marketing = pd.DataFrame({
    "channel": rng.choice(["搜索", "社交", "会员"], n),
    "visits": rng.integers(80, 850, n),
    "ad_spend": rng.uniform(2, 38, n),
})
marketing["sales"] = (
    45 + marketing["visits"] * 0.16 + marketing["ad_spend"] * 2.4
    + marketing["channel"].map({"搜索": 18, "社交": 8, "会员": 32})
    + rng.normal(0, 28, n)
).clip(10)
marketing["conversion"] = (marketing["sales"] / marketing["visits"]).clip(0.02, 0.5)

daily = pd.DataFrame({
    "date": np.tile(pd.date_range("2026-01-01", periods=12, freq="D"), 3),
    "region": np.repeat(["华东", "华南", "华北"], 12),
})
daily["sales"] = (
    np.tile(np.linspace(110, 190, 12), 3)
    + np.repeat([28, 8, 18], 12)
    + rng.normal(0, 9, 36)
)

sns.set_theme(style="whitegrid", context="notebook")
print("订单样本:", orders.shape, "营销样本:", marketing.shape)


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
corr = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("营销指标相关系数")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
pivot = orders.pivot_table(index="region", columns="category", values="order_value", aggfunc="mean")
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", linewidths=0.5, cbar_kws={"label": "平均客单价（元）"}, ax=ax)
ax.set(title="区域与品类客单价", xlabel="品类", ylabel="区域")
fig.tight_layout()
plt.show()


## 3. 参数说明

- annot：标注
- fmt：格式
- cmap：色盘
- center/vmin/vmax：色阶


## 4. 结果解读

先读色阶含义，再找极值、带状结构和异常组合。


## 常见误区

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。


In [ ]:
counts = pd.crosstab(orders["region"], orders["channel"])
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(counts, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set(title="区域与渠道订单量", xlabel="渠道", ylabel="区域")
fig.tight_layout()
plt.show()


## 本章小结

用颜色矩阵展示相关系数、透视表或任意二维数值。


### 你已经掌握

- 判断热力图（heatmap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较行×列组合或压缩读取数值矩阵。 |
| 数据结构 | 二维矩阵或可透视成长×宽矩阵的长表。 |
| 结果解读 | 先读色阶含义，再找极值、带状结构和异常组合。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `annot` | 标注 |
| `fmt` | 格式 |
| `cmap` | 色盘 |
| `center/vmin/vmax` | 色阶 |


### 需要注意

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


### 完成检查

- [ ] 能判断什么问题适合使用热力图（heatmap）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论
